# Rule validation, mechanism reconstruction, and counterexamples

This notebook consumes only successfully completed prior phases with the same source SHA-256. It recomputes constraints and key prior findings, tests stability on disjoint random and chronological partitions, records counterexamples, and reconciles every earlier candidate without changing the original registers. Implementation is in `validation_analysis`, `fit_gap_model`, `endpoint_validation`, and `build_rule_register` in `src/remaining_analysis.py`.

Random partitions use seed 2026. The chronological cut is the median observed epoch: early includes ties, late excludes them. These are internal stability checks on a synthetic population already explored by earlier notebooks, not an untouched external validation set. Only the gap mechanism has development-fitted parameters evaluated without refitting on the random validation rows. Related transactions can cross partitions; no predictive accuracy claim follows.

Confidence labels distinguish high descriptive confidence about observed structural constraints from moderate evidence about an unidentified generating mechanism or adjusted association. Zero exceptions in this population do not prove universality.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown, Image

ROOT = Path.cwd().resolve()
if not (ROOT / "data/cleaned/swift_payments_cleaned.csv").exists():
    ROOT = ROOT.parent
assert (ROOT / ".venv/Scripts/python.exe").exists()
assert Path(sys.executable).resolve() == (ROOT / ".venv/Scripts/python.exe").resolve(), "Use the project virtual environment"
sys.path.insert(0, str(ROOT / "src"))
from remaining_analysis import Study, validation_analysis
study = Study(ROOT, '08_rule_validation', prerequisites=('05_currency_party', '06_anomaly_analysis', '07_tokens'))

## Structural constraints and reproduction

**Purpose.** Ensure proposed hard rules and earlier headline findings survive independent recalculation.

**Hypothesis.** Direct routing and participant count are equivalent structural branches, and FX never exceeds participants minus one.

**Methodology.** Enumerate all eligible rows and exceptions for five structural rules in full, random-development, random-validation, chronological-early and chronological-late populations. Wilson intervals refer only to hypothetical generator draws. Recompute overall and route medians; reproduce the prior Covered-vs-Indirect estimator at minimum one row per arm before applying a stricter minimum of 20.

**Evidence.** The following code displays calculated tables. Counts are transaction denominators unless stated otherwise. All duration values use seconds.

**Interpretation.** A strict rule is supported in the observed population only if exceptions are zero. Exact reproduction validates calculation consistency, not economic causality.

**Limitations.** Routing labels may encode definitions. The earlier minimum-one estimator includes small cells; the revised estimate has a narrower common-support estimand.

**Business Implications.** Use validated constraints as synthetic-data consistency checks, with explicit populations and exception counts.

In [ ]:
structural, mechanisms, split_effects, endpoints, fx_comparisons, gap_comparisons, registry, reconciliation = validation_analysis(study)
display(structural)
display(pd.read_csv(study.tables / "prior_route_median_reproduction.csv"))
display(pd.read_csv(study.tables / "prior_matched_duration_reproduction.csv"))

## Gap and endpoint generating mechanisms

**Purpose.** Distinguish an approximate stochastic mechanism from a hard business rule.

**Hypothesis.** At-least-one-gap rates approximately follow 1-(1-p)^(participants-1), and endpoint mismatches approximately follow independent marginal perturbations.

**Methodology.** Fit a binomial likelihood to participant-specific gap counts, report model-based Fisher-information intervals for p and residual errors in percentage points. Fit the random-development subset and evaluate fixed predictions on random-validation rows. Repeat parameter estimates chronologically. Rescan visible BIC endpoints read-only and compare joint mismatches to the product of marginal rates, reporting Cramér V rather than treating a nonsignificant p-value as proof of independence.

**Evidence.** The following code displays calculated tables. Counts are transaction denominators unless stated otherwise. All duration values use seconds.

**Interpretation.** Close calibration is compatible with a per-link generator, but aggregate any-gap flags cannot identify independence at the individual link level. Nonzero residuals remain evidence against exact calibration.

**Limitations.** Fisher intervals assume the fitted mechanism and independent transactions. Sparse participant strata and an unobserved link process limit identification. Endpoint mismatch can be synthetic perturbation rather than operational error.

**Business Implications.** Calibrate traceability expectations to path complexity; do not equate perturbed identifiers with slow or failed payments.

In [ ]:
display(mechanisms)
display(endpoints)
display(pd.read_csv(study.tables / "gap_heldout_calibration.csv"))
display(split_effects)

## Rejected hypotheses and complete candidate disposition

**Purpose.** Prevent selective reporting and unsupported rule promotion.

**Hypothesis.** More FX and routing gaps always imply longer median durations; all prior candidate hypotheses are fully validated.

**Methodology.** Recompute adjacent-FX and gap-versus-no-gap medians within route/participant strata (and FX for gaps), requiring 100 rows per arm. Report both effect directions. Reconcile all 34 earlier candidates to supported, revised, rejected, or exploratory dispositions; related evidence is explicitly not equivalent to exact-hypothesis validation. New rules include condition, outcome, denominator, baseline, effect, uncertainty, exceptions, confounders, robustness, confidence and business implication.

**Evidence.** The following code displays calculated tables. Counts are transaction denominators unless stated otherwise. All duration values use seconds.

**Interpretation.** One observed counterexample defeats a universal rule. Mixed directions require conditional explanations. Candidates without independent support remain exploratory; none are silently marked complete.

**Limitations.** Exact medians are finite-population summaries, not confidence intervals for future payments. Multiple comparisons and residual composition affect subgroup interpretation. BIC-pair, temporal and several party hypotheses remain open.

**Business Implications.** A defensible rule registry prioritizes what can be checked, what must be revised, and what needs further evidence.

In [ ]:
display(fx_comparisons)
display(gap_comparisons)
display(registry)
display(reconciliation)
display(Image(filename=str(study.charts / "validated_route_medians.png")))

In [ ]:
report = study.finish([f"Structural validation: {len(structural)} rule-by-split tests; total exceptions across reported tests = {int(structural.exceptions.sum())} (partitions overlap full population and should not be added as independent evidence).",
                       f"Full-population per-link p = {mechanisms.loc[mechanisms.split.eq('full'), 'per_link_probability'].iloc[0]:.9f}.",
                       f"Adequate adjacent-FX comparisons: {int(((fx_comparisons.difference_seconds<0)&fx_comparisons.adequate).sum())} negative of {int(fx_comparisons.adequate.sum())}.",
                       f"Gap median comparisons: {int((gap_comparisons.difference_seconds<0).sum())} negative and {int((gap_comparisons.difference_seconds>0).sum())} positive of {len(gap_comparisons)}.",
                       f"Registry contains {len(registry)} scoped rules; all {len(reconciliation)} earlier candidates reconciled. Unvalidated hypotheses remain marked exploratory."], ['Internal random/time splits do not establish real-world transportability or causality.', 'Independent-link generation is plausible but not uniquely identified.', 'Pointwise intervals and split checks do not remove multiple-testing or related-record dependence.', 'BIC-pair, time-of-day and several original party hypotheses remain open; their dispositions are explicit.'], 'Prepare aggregate dashboard data with reconciliation, a data dictionary, suppression accounting and nonadditive-metric warnings.')
display(Markdown("### Final findings and limitations\n" + "\n".join("- " + x for x in report["findings"]) + "\n\n" + " ".join(report["limitations"])))
display(pd.DataFrame(study.checks))
display(Markdown("### Completion summary\nExecution passed for " + format(report["population"], ",") + " transactions. All generated files are listed below. No unresolved material errors.\n\n" + "\n".join("- `"+x+"`" for x in report["files"]) + "\n\nRecommended next investigation: " + report["next_investigation"]))